# バーンスタイン・ヴァジラニのアルゴリズム(概要)

バーンスタイン・ヴァジラニのアルゴリズムについて説明します。

このアルゴリズムでは、$x$ を入力として受け取り、内部のビット列 $a$ を使って出力を計算する以下の関数を考えます。

$$
f_a(x) = (a\cdot x) \bmod2 = (\sum_i a_i x_i) \bmod2
$$

目標は、$f_a(x)$ の出力からビット列 $a$ を求めることです。

$a, x$ はそれぞれ $n$ ビットの文字列 $a = a_0 a_1... a_{n-1}$、$x = x_0 x_1.... x_{n-1}$ です。

$a$ が4ビットの列 $1001$ である場合を考えます。
古典計算で確実に $x$ を求める方法は、$x=1000, 0100, 0010, 0001$ のように、1ビットだけを1にして残りを0にした $x$ を順番に入力することです。
$0 \cdot 0 = 0 \cdot 1 = 0$ であることから、$x_i=0$ のビットは結果に影響しないため、1ビットずつ $a_i$ の値を決定できます。

$a=1001$ について、4通りの異なる $x$ に対して $f_a(x)$ を計算すると以下のようになります。

$(1001 \cdot 1000) \bmod2 = 1$  
$(1001 \cdot 0100) \bmod2 = 0$  
$(1001 \cdot 0010) \bmod2 = 0$  
$(1001 \cdot 0001) \bmod2 = 1$

上記の出力から、$a=1001$ であることが分かります。

バーンスタイン・ヴァジラニのアルゴリズムは、量子重ね合わせを使って一度の計算で $a$ を求めます。

$a=1001$ の場合の具体的な回路は以下の通りです。

<img src="img/102_img.png" width="30%">

$a_i=1$ である量子ビット $i$ を制御ビットとして $CX$ ゲートを作用させます。
ターゲットビットはすべて4番目の補助量子ビットです。

状態を確認してみましょう。

$$
\begin{align}
\lvert \psi_1\rangle &= \biggl(\otimes^4 H\lvert 0\rangle \biggr)\otimes H \lvert 1\rangle \\
&= \frac{1}{\sqrt{2}}(\lvert 0\rangle + \lvert 1\rangle) \otimes \frac{1}{\sqrt{2}}(\lvert 0\rangle + \lvert 1\rangle) \otimes \frac{1}{\sqrt{2}}(\lvert 0\rangle + \lvert 1\rangle) \otimes \frac{1}{\sqrt{2}}(\lvert 0\rangle + \lvert 1\rangle) \otimes \frac{1}{\sqrt{2}}(\lvert 0\rangle - \lvert 1\rangle)
\end{align}
$$

次に $\lvert \psi_2\rangle$ の状態を確認しましょう。
$CX$ ゲートが作用する0番目と4番目の量子ビットだけに注目します。

$$
\begin{align}
\lvert \psi_1\rangle_{04} &= \frac{1}{\sqrt{2}}(\lvert 0\rangle_0 + \lvert 1\rangle_0) \otimes \frac{1}{\sqrt{2}}(\lvert 0\rangle_4 - \lvert 1\rangle_4) \\
&= \lvert +\rangle_0 \otimes \lvert -\rangle_4
\end{align}
$$

$$
\begin{align}
\lvert \psi_2\rangle_{04} &= \frac{1}{\sqrt{2}}\lvert 0\rangle_0 \otimes \frac{1}{\sqrt{2}}(\lvert 0\rangle_4 - \lvert 1\rangle_4) + \frac{1}{\sqrt{2}}\lvert 1\rangle_0 \otimes \frac{1}{\sqrt{2}}(\lvert 1\rangle_4 - \lvert 0\rangle_4) \\
&= \frac{1}{\sqrt{2}}\lvert 0\rangle_0 \otimes \frac{1}{\sqrt{2}}(\lvert 0\rangle_4 - \lvert 1\rangle_4) - \frac{1}{\sqrt{2}}\lvert 1\rangle_0 \otimes \frac{1}{\sqrt{2}}(\lvert 0\rangle_4 - \lvert 1\rangle_4) \\
&= \frac{1}{\sqrt{2}}(\lvert 0\rangle_0 - \lvert 1\rangle_0 ) \otimes \frac{1}{\sqrt{2}}(\lvert 0\rangle_4 - \lvert 1\rangle_4) \\
&=  \lvert -\rangle_0 \otimes \lvert -\rangle_4
\end{align}
$$

0番目と4番目の量子ビットに $CX$ ゲートを作用させることで、0番目の量子ビットの位相が反転し、$\lvert + \rangle \to \lvert - \rangle$ となりました。
同様に、3番目と4番目の量子ビットに $CX$ ゲートを作用させると、状態 $\lvert\psi_3\rangle$ は以下のようになります。

$$
\lvert \psi_3\rangle = \frac{1}{\sqrt{2}}(\lvert 0\rangle - \lvert 1\rangle) \otimes \frac{1}{\sqrt{2}}(\lvert 0\rangle + \lvert 1\rangle) \otimes \frac{1}{\sqrt{2}}(\lvert 0\rangle + \lvert 1\rangle) \otimes \frac{1}{\sqrt{2}}(\lvert 0\rangle - \lvert 1\rangle) \otimes \frac{1}{\sqrt{2}}(\lvert 0\rangle - \lvert 1\rangle)
$$

最後に、Hゲートを作用させます。

$$
\lvert \psi_4\rangle = \lvert 1\rangle \otimes \lvert 0\rangle \otimes \lvert 0\rangle \otimes \lvert 1\rangle \otimes \frac{1}{\sqrt{2}}(\lvert 0\rangle - \lvert 1\rangle)
$$

0番目から3番目までの量子ビットを測定すると、測定結果として $1001$ が得られます。
これで $a$ を求めることができました。

Blueqatでこれを実装してみましょう。

In [1]:
from blueqat import Circuit
import numpy as np
from collections import Counter as _Counter

# Compatibility note: the new blueqat SDK reports measurement bitstrings with
# qubit 0 as the right-most character (standard binary ordering), while this
# notebook (like the old blueqat 0.3.x) reads qubit 0 as the left-most
# character. Patch Circuit.run once so sampled bitstrings keep matching the
# qubit ordering used throughout this notebook's explanation.
if not getattr(Circuit, '_qubit0_leftmost_patched', False):
    _original_run = Circuit.run
    def _run_qubit0_leftmost(self, *args, **kwargs):
        result = _original_run(self, *args, **kwargs)
        if isinstance(result, _Counter):
            return _Counter({key[::-1]: count for key, count in result.items()})
        return result
    Circuit.run = _run_qubit0_leftmost
    Circuit._qubit0_leftmost_patched = True

まず、オラクル $U_f$ を作用させる関数を用意します。

In [2]:
def Uf(c, a):
    N = len(a)
    for i, val in enumerate(list(a)):
        if val == '1':
            c.cx[i, len(a)]

以下がアルゴリズムの本体です。
まず、求めたい $a$ を乱数で決定します。

オラクルを使った量子回路の出力結果から $a$ を求め、答えが正しいかどうかを確認します。

In [3]:
n = 4

a = ''
for i in range(n):
    a += str(np.random.randint(2))
    
c = Circuit(n + 1)
c.x[n].h[:]
Uf(c, a)
c.h[:].m[:]
res = c.run(shots = 1000)

print(res)

if [arr[:n] for arr in res.keys()] == [a]:
    print("OK")
else:
    print("incorrect")

Counter({'00001': 1000})
OK


以上により、バーンスタイン・ヴァジラニのアルゴリズムを使って、オラクルが内部に持つ $a$ を求めることができました。